In [ ]:
from IPython.core.magic import register_line_magic
import importlib
import subprocess
import sys
from packaging import version as pkg_version
from IPython.display import display, HTML
import os

# Internal queue for pending installs
_pending_packages = []

@register_line_magic
def checkpip(line):
    """
    Usage:
      %checkpip package
      %checkpip package>=x.y.z
      %checkpip --apply    -> installs all queued packages and restarts kernel

    Queues installs/upgrades quietly until you run --apply.
    """
    global _pending_packages

    # Apply all queued installs
    if line.strip() == "--apply":
        if not _pending_packages:
            print("✅ No packages to install/upgrade.")
            return
        print(f"Installing/upgrading {_pending_packages}...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet"] + _pending_packages,
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        display(HTML("<b style='color: red;'>Restarting kernel to apply package changes...</b>"))
        os._exit(0)  # Restart kernel immediately
        return

    # Normal package check
    if ">=" in line:
        pkg_name, min_ver = line.split(">=", 1)
        pkg_name, min_ver = pkg_name.strip(), min_ver.strip()
    else:
        pkg_name, min_ver = line.strip(), None

    try:
        pkg = importlib.import_module(pkg_name)
        if min_ver and pkg_version.parse(pkg.__version__) < pkg_version.parse(min_ver):
            _pending_packages.append(f"{pkg_name}>={min_ver}")
            print(f"⏳ Queued upgrade: {pkg_name} from {pkg.__version__} to >= {min_ver}")
    except ImportError:
        _pending_packages.append(f"{pkg_name}>={min_ver}" if min_ver else pkg_name)
        print(f"⏳ Queued install: {pkg_name} (>= {min_ver})" if min_ver else f"⏳ Queued install: {pkg_name}")


In [ ]:
%checkpip pip
%checkpip nbformat
%checkpip os
%checkpip packaging
%checkpip IPython
%checkpip numpy
%checkpip matplotlib
%checkpip ipywidgets
%checkpip scipy
%checkpip pyentrp
%checkpip subprocess
%checkpip platform
%checkpip pyserial
%checkpip torch
%checkpip transformers
%checkpip --apply